# Rideshare Analytics — Analysis Notebook

Interactive exploration of the gold-layer marts and the Dynamic-Tables-vs-Incremental experiment.

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import snowflake.connector

load_dotenv(Path('..').resolve() / '.env')
sns.set_theme(style='whitegrid')

def conn():
    return snowflake.connector.connect(
        account=os.environ['SNOWFLAKE_ACCOUNT'],
        user=os.environ['SNOWFLAKE_USER'],
        password=os.environ['SNOWFLAKE_PASSWORD'],
        role=os.environ.get('SNOWFLAKE_ROLE','DBT_ROLE'),
        warehouse=os.environ.get('SNOWFLAKE_WAREHOUSE','DBT_WH_XS'),
        database=os.environ.get('SNOWFLAKE_DATABASE','RIDESHARE_DB'),
    )

def q(sql):
    c = conn(); cur = c.cursor(); cur.execute(sql)
    cols = [d[0].lower() for d in cur.description]
    df = pd.DataFrame(cur.fetchall(), columns=cols)
    cur.close(); c.close(); return df

## Operations: driver utilization (BASELINE)

In [ ]:
q('SELECT utilisation_band, COUNT(*) AS drivers FROM MARTS.MART_DRIVER_UTILIZATION_BASELINE GROUP BY 1')

## Compare BASELINE vs INTERVENTION

In [ ]:
b = q('SELECT COUNT(*) AS n, AVG(trips_24h) AS avg_trips FROM MARTS.MART_DRIVER_UTILIZATION_BASELINE')
i = q('SELECT COUNT(*) AS n, AVG(trips_24h) AS avg_trips FROM STREAMS_DT.MART_DRIVER_UTILIZATION_DT')
print('baseline:', b.iloc[0].to_dict())
print('intervention:', i.iloc[0].to_dict())

## Experiment results

In [ ]:
m = q('SELECT * FROM META.experiment_metrics')
m.head()

In [ ]:
m.groupby(['condition','metric_name'])['metric_value'].agg(['mean','std','count']).reset_index()

In [ ]:
wide = m.pivot_table(index=['run_uuid','metric_name'], columns='condition',
                    values='metric_value', aggfunc='first').reset_index()
rows = []
for metric in ['freshness_lag_seconds', 'credits_used']:
    sub = wide[wide['metric_name']==metric].dropna(subset=['BASELINE_INCREMENTAL','INTERVENTION_DYNAMIC_TABLE'])
    b = sub['BASELINE_INCREMENTAL']; i = sub['INTERVENTION_DYNAMIC_TABLE']
    diff = (i - b).values
    if len(diff) < 5: continue
    w_stat, p = stats.wilcoxon(i, b)
    rows.append({'metric': metric, 'n': len(sub),
                 'baseline_mean': b.mean(), 'intervention_mean': i.mean(),
                 'mean_diff': diff.mean(), 'wilcoxon_p': p,
                 'cohens_d': diff.mean() / (diff.std(ddof=1)+1e-12)})
pd.DataFrame(rows)